# Simulador de Rutas Reales con API OSRM

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** IA & Big Data · APIs & Visualización · Geolocalización

---

### Objetivo

Calcular la ruta real en coche entre dos puntos geográficos usando el servicio open-source
**OSRM** (Open Source Routing Machine) y simular el recorrido con una animación sobre un mapa
interactivo, construido con **ipyleaflet** e **ipywidgets**.

### Contexto de negocio

**El cliente:** cualquier app de movilidad, delivery o logística que necesite mostrar al
usuario no una estimación abstracta, sino la ruta real que recorrerá un vehículo.

**El problema:** la distancia en línea recta entre dos coordenadas no sirve para decisiones
operativas — un repartidor no vuela, tiene que seguir las carreteras. Calcular eso a mano, o
con una librería de geometría genérica, no tiene en cuenta el callejero real.

**La pregunta:** dado un origen y un destino, ¿cuál es la ruta real, cuánto mide, cuánto se
tarda, y cómo se comunica eso de forma visual e intuitiva para quien no es analista de datos?

El usuario hace clic en el mapa para elegir origen y destino; la aplicación calcula la ruta
real (no la línea recta), interpola los puntos para una animación fluida y anima un marcador
recorriendo la ruta a la velocidad que se configure.

In [1]:
import time
import requests
import numpy as np

import ipywidgets as widgets
from ipywidgets import Button, HBox, VBox, Output, FloatSlider, HTML

from ipyleaflet import (
    Map,
    Marker,
    CircleMarker,
    Polyline,
    basemaps
)

## 1. Cálculo de la ruta con la API de OSRM

In [2]:
def calcular_ruta_coche(origen, destino):
    """
    Calcula una ruta en coche utilizando la API de OSRM.

    Parámetros:
        origen (tuple): Coordenadas de origen en formato (lat, lon).
        destino (tuple): Coordenadas de destino en formato (lat, lon).

    Devuelve:
        tuple: distancia_km, duracion_min, lista_de_puntos [(lat, lon), ...]
    """

    lat1, lon1 = origen
    lat2, lon2 = destino

    # Construir la URL para la API de OSRM
    url = (
        f"http://router.project-osrm.org/route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=full&geometries=geojson"
    )

    # Realizar la petición
    respuesta = requests.get(url, timeout=10)
    respuesta.raise_for_status()

    # Obtener los datos en formato JSON
    data = respuesta.json()

    # Comprobar que la ruta se ha calculado correctamente
    if data.get("code") != "Ok" or not data.get("routes"):
        raise ValueError(
            "No se pudo calcular la ruta. Verifique las coordenadas."
        )

    # Obtener la primera ruta
    ruta = data["routes"][0]

    # Convertir metros a kilómetros
    distancia_km = ruta["distance"] / 1000

    # Convertir segundos a minutos
    duracion_min = ruta["duration"] / 60

    # OSRM devuelve las coordenadas como [lon, lat].
    # Las convertimos a (lat, lon) para ipyleaflet.
    coords_geojson = ruta["geometry"]["coordinates"]
    puntos_ruta = [(lat, lon) for lon, lat in coords_geojson]

    return distancia_km, duracion_min, puntos_ruta

## 2. Interpolación de la ruta para una animación fluida

In [3]:
def haversine_km(p1, p2):
    """
    Calcula la distancia en kilómetros entre dos puntos geográficos
    utilizando la fórmula de Haversine.

    Parámetros:
        p1, p2: Coordenadas (lat, lon) en grados de cada punto.

    Devuelve:
        float: Distancia en kilómetros.
    """
    lat1, lon1 = np.radians(p1)
    lat2, lon2 = np.radians(p2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 6371 * 2 * np.arcsin(np.sqrt(a))


def interpolar_ruta(puntos_ruta, n_pasos=250):
    """
    Genera n_pasos puntos equiespaciados por distancia recorrida
    a lo largo de la ruta, para animar el recorrido con velocidad
    constante en vez de con velocidad proporcional al número de
    vértices que devuelve la API.
    """
    coords = np.array(puntos_ruta)

    # Distancia acumulada entre los puntos
    distancias = [0.0]
    for i in range(1, len(coords)):
        distancias.append(distancias[-1] + haversine_km(coords[i - 1], coords[i]))
    distancias = np.array(distancias)

    distancias_totales = distancias[-1]
    nuevas_distancias = np.linspace(0, distancias_totales, n_pasos)
    lats_interp = np.interp(nuevas_distancias, distancias, coords[:, 0])
    lons_interp = np.interp(nuevas_distancias, distancias, coords[:, 1])

    return list(zip(lats_interp, lons_interp))

## 3. Mapa interactivo con ipyleaflet

Interfaz: el usuario hace clic en el mapa para fijar origen y destino, la app calcula y
dibuja la ruta real, y un botón lanza la animación del recorrido a la velocidad elegida
con el slider.

In [4]:
# Estados de la aplicación
estado = {
    "puntos": [],
    "marcadores": [],
    "linea_ruta": None,
    "puntos_animacion": None,
    "bolita": None,
}


# Crear el mapa
mapa = Map(
    center=(40.4168, -3.7038),
    zoom=5,
    basemap=basemaps.OpenStreetMap.Mapnik
)

mapa.layout.height = "600px"


# Elementos de la interfaz
salida = Output()

etiqueta_info = HTML(
    value="<b>Haz clic en el mapa para elegir el punto de origen.</b>"
)

boton_reiniciar = Button(
    description="Reiniciar",
    button_style="warning",
    icon="refresh"
)

boton_animar = Button(
    description="Iniciar animación",
    button_style="success",
    icon="play",
    disabled=True
)

slider_velocidad = FloatSlider(
    value=0.05,
    min=0.01,
    max=0.1,
    step=0.005,
    description="Velocidad (s/paso)"
)


# ==========================================
# FUNCIÓN PARA LIMPIAR EL MAPA
# ==========================================

def limpiar_mapa():
    # Eliminar todas las capas excepto la capa base
    for capa in list(mapa.layers):
        if capa is not mapa.layers[0]:
            mapa.remove_layer(capa)

    # Reiniciar el estado
    estado["puntos"] = []
    estado["marcadores"] = []
    estado["linea_ruta"] = None
    estado["puntos_animacion"] = None
    estado["bolita"] = None

    # Desactivar el botón de animación
    boton_animar.disabled = True

    # Restablecer el mensaje
    etiqueta_info.value = (
        "<b>Haz clic en el mapa para elegir el punto de origen.</b>"
    )


# ==========================================
# FUNCIÓN PARA MANEJAR LOS CLICS EN EL MAPA
# ==========================================

def manejar_click(**kwargs):
    # Comprobar que se trata de un clic
    if kwargs.get("type") != "click":
        return

    # Solo permitir seleccionar dos puntos
    if len(estado["puntos"]) >= 2:
        return

    # Obtener coordenadas del clic
    lat, lon = kwargs["coordinates"]

    # Guardar el punto
    estado["puntos"].append((lat, lon))

    # Crear el marcador
    marcador = Marker(
        location=(lat, lon),
        draggable=False
    )

    mapa.add_layer(marcador)
    estado["marcadores"].append(marcador)

    # Actualizar la interfaz según el punto seleccionado
    if len(estado["puntos"]) == 1:
        etiqueta_info.value = (
            "<b>Origen seleccionado.</b> "
            "Haz clic en el mapa para elegir el destino."
        )

    elif len(estado["puntos"]) == 2:
        etiqueta_info.value = (
            "<b>Calculando ruta en coche...</b>"
        )

        calcular_y_dibujar_ruta()


# ==========================================
# CALCULAR Y DIBUJAR LA RUTA
# ==========================================

def calcular_y_dibujar_ruta():
    origen, destino = estado["puntos"]

    with salida:
        try:
            # Calcular la ruta
            distancia_km, duracion_min, puntos_ruta = (
                calcular_ruta_coche(origen, destino)
            )

        except Exception as e:
            etiqueta_info.value = (
                f"<b style=\'color:red\'>"
                f"Error al calcular la ruta:</b> {e}"
            )
            return

    # Dibujar la línea de la ruta
    linea = Polyline(
        locations=puntos_ruta,
        color="#2255cc",
        weight=4,
        fill=False
    )

    mapa.add_layer(linea)
    estado["linea_ruta"] = linea

    # Crear los puntos para la animación
    estado["puntos_animacion"] = interpolar_ruta(
        puntos_ruta,
        n_pasos=250
    )

    # Mostrar información de la ruta
    etiqueta_info.value = (
        f"<b>Distancia en coche:</b> {distancia_km:.1f} km "
        f"&nbsp;|&nbsp; "
        f"<b>Tiempo estimado:</b> {duracion_min:.0f} min"
    )

    # Activar el botón de animación
    boton_animar.disabled = False


# ==========================================
# FUNCIÓN PARA ANIMAR LA RUTA
# ==========================================

def iniciar_animacion(b):
    # Comprobar que existe una ruta
    if not estado["puntos_animacion"]:
        return

    # Eliminar la bolita anterior si existe
    if estado["bolita"] is not None:
        mapa.remove_layer(estado["bolita"])

    # Crear la bolita
    bolita = CircleMarker(
        location=estado["puntos_animacion"][0],
        radius=8,
        color="red",
        fill_color="red",
        fill_opacity=0.9
    )

    mapa.add_layer(bolita)
    estado["bolita"] = bolita

    # Desactivar el botón mientras se ejecuta la animación
    boton_animar.disabled = True

    # Animar la bolita
    for punto in estado["puntos_animacion"]:
        bolita.location = punto
        time.sleep(slider_velocidad.value)

    # Volver a activar el botón
    boton_animar.disabled = False


# ==========================================
# ASIGNAR EVENTOS
# ==========================================

boton_reiniciar.on_click(
    lambda b: limpiar_mapa()
)

boton_animar.on_click(
    iniciar_animacion
)

mapa.on_interaction(
    manejar_click
)


# ==========================================
# CREAR EL LAYOUT
# ==========================================

controles = VBox([
    etiqueta_info,
    HBox([
        boton_reiniciar,
        boton_animar,
        slider_velocidad
    ])
])


# Mostrar mapa y controles
VBox([
    mapa,
    controles,
    salida
])

## 4. Conclusión

**Lo que resuelve este notebook:**
- Una API de rutas **gratuita y sin API key** (OSRM) es suficiente para obtener geometría de
  carretera real, distancia y tiempo estimado — no hace falta un proveedor de pago para un
  prototipo o un MVP.
- Interpolar **por distancia recorrida** (Haversine) en vez de por número de vértices es lo que
  hace que la animación se mueva a velocidad constante, independientemente de cómo de detallada
  sea la geometría que devuelve la API en cada tramo.
- Con `ipyleaflet` + `ipywidgets`, un notebook deja de ser solo un informe estático y se
  convierte en un prototipo interactivo — útil para validar una idea de producto (p. ej. una
  app de delivery) antes de invertir en una interfaz web completa.

**Aplicabilidad:** el mismo patrón (API de rutas + interpolación + animación) sirve para
simular flotas de reparto, estimar SLAs de entrega por zona, o construir un demo de producto
para stakeholders no técnicos sin montar una aplicación web desde cero.